In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Men's data
m_reg = pd.read_csv(
    RAW_DIR / "MRegularSeasonDetailedResults.csv"
)

m_tourney = pd.read_csv(
    RAW_DIR / "MNCAATourneyCompactResults.csv"
)

m_seeds = pd.read_csv(
    RAW_DIR / "MNCAATourneySeeds.csv"
)

# Women's data
w_reg = pd.read_csv(
    RAW_DIR / "WRegularSeasonDetailedResults.csv"
)

w_tourney = pd.read_csv(
    RAW_DIR / "WNCAATourneyCompactResults.csv"
)

w_seeds = pd.read_csv(
    RAW_DIR / "WNCAATourneySeeds.csv"
)

print("Men regular season:", m_reg.shape)
print("Men tournament:", m_tourney.shape)
print("Men seeds:", m_seeds.shape)

print()

print("Women regular season:", w_reg.shape)
print("Women tournament:", w_tourney.shape)
print("Women seeds:", w_seeds.shape)

Men regular season: (124529, 34)
Men tournament: (2585, 8)
Men seeds: (2694, 3)

Women regular season: (87187, 34)
Women tournament: (1717, 8)
Women seeds: (1812, 3)


In [4]:
def create_team_game_rows(df):

    # Winner's perspective
    winners = pd.DataFrame({
        "Season": df["Season"],
        "DayNum": df["DayNum"],

        "TeamID": df["WTeamID"],
        "OppTeamID": df["LTeamID"],

        "TeamScore": df["WScore"],
        "OppScore": df["LScore"],

        "Won": 1,

        # Team shooting
        "FGM": df["WFGM"],
        "FGA": df["WFGA"],
        "FGM3": df["WFGM3"],

        # Opponent shooting
        "OppFGM": df["LFGM"],
        "OppFGA": df["LFGA"],
        "OppFGM3": df["LFGM3"]
    })

    # Loser's perspective
    losers = pd.DataFrame({
        "Season": df["Season"],
        "DayNum": df["DayNum"],

        "TeamID": df["LTeamID"],
        "OppTeamID": df["WTeamID"],

        "TeamScore": df["LScore"],
        "OppScore": df["WScore"],

        "Won": 0,

        # Team shooting
        "FGM": df["LFGM"],
        "FGA": df["LFGA"],
        "FGM3": df["LFGM3"],

        # Opponent shooting
        "OppFGM": df["WFGM"],
        "OppFGA": df["WFGA"],
        "OppFGM3": df["WFGM3"]
    })

    # One original game becomes two team-perspective rows
    team_games = pd.concat(
        [winners, losers],
        ignore_index=True
    )

    return team_games

m_games = create_team_game_rows(m_reg)
w_games = create_team_game_rows(w_reg)

print("Men team-game rows:", m_games.shape)
print("Women team-game rows:", w_games.shape)

display(m_games.head())

Men team-game rows: (249058, 13)
Women team-game rows: (174374, 13)


,Season,DayNum,TeamID,OppTeamID,TeamScore,OppScore,Won,FGM,FGA,FGM3,OppFGM,OppFGA,OppFGM3
0,2003,10,1104,1328,68,62,1,27,58,3,22,53,2
1,2003,10,1272,1393,70,63,1,26,62,8,24,67,6
2,2003,11,1266,1437,73,61,1,24,58,8,22,73,3
3,2003,11,1296,1457,56,50,1,18,38,3,18,49,6
4,2003,11,1400,1208,77,71,1,30,61,6,24,62,6


In [5]:
assert len(m_games) == 2 * len(m_reg)
assert len(w_games) == 2 * len(w_reg)

print("Restructuring check passed.")

Restructuring check passed.


In [6]:
m_games["ScoringMargin"] = (
    m_games["TeamScore"] - m_games["OppScore"]
)

w_games["ScoringMargin"] = (
    w_games["TeamScore"] - w_games["OppScore"]
)

In [7]:
def create_team_season_stats(games):

    stats = (
        games
        .groupby(["Season", "TeamID"])
        .agg(
            # Basic season information
            GamesPlayed=("Won", "count"),
            Wins=("Won", "sum"),

            # Scoring
            AvgScoringMargin=("ScoringMargin", "mean"),

            # Team shooting totals
            FGM=("FGM", "sum"),
            FGA=("FGA", "sum"),
            FGM3=("FGM3", "sum"),

            # Opponent shooting totals
            OppFGM=("OppFGM", "sum"),
            OppFGA=("OppFGA", "sum"),
            OppFGM3=("OppFGM3", "sum")
        )
        .reset_index()
    )

    # -------------------------
    # Feature component 1:
    # Win Rate
    # -------------------------

    stats["WinRate"] = (
        stats["Wins"] /
        stats["GamesPlayed"]
    )

    # -------------------------
    # Feature component 2:
    # Offensive eFG%
    # -------------------------

    stats["EFG"] = (
        stats["FGM"] +
        0.5 * stats["FGM3"]
    ) / stats["FGA"]

    # -------------------------
    # Feature component 3:
    # Opponent eFG%
    # -------------------------

    stats["OpponentEFG"] = (
        stats["OppFGM"] +
        0.5 * stats["OppFGM3"]
    ) / stats["OppFGA"]

    # -------------------------
    # eFG Differential
    # -------------------------

    stats["EFGDifferential"] = (
        stats["EFG"] -
        stats["OpponentEFG"]
    )

    return stats

m_stats = create_team_season_stats(m_games)
w_stats = create_team_season_stats(w_games)

print("Men team-seasons:", m_stats.shape)
print("Women team-seasons:", w_stats.shape)

display(m_stats.head())

Men team-seasons: (8346, 15)
Women team-seasons: (5965, 15)


,Season,TeamID,GamesPlayed,Wins,AvgScoringMargin,FGM,FGA,FGM3,OppFGM,OppFGA,OppFGM3,WinRate,EFG,OpponentEFG,EFGDifferential
0,2003,1102,28,12,0.250000,536,1114,219,540,1188,133,0.428571,0.579443,0.510522,0.068922
1,2003,1103,27,13,0.629630,733,1508,147,750,1539,180,0.481481,0.534814,0.545809,-0.010995
2,2003,1104,28,17,4.285714,673,1601,178,651,1554,178,0.607143,0.475953,0.476190,-0.000238
3,2003,1105,26,7,-4.884615,634,1602,197,702,1533,163,0.269231,0.457241,0.511089,-0.053848
4,2003,1106,28,13,-0.142857,656,1548,171,608,1495,134,0.464286,0.479005,0.451505,0.027500


In [8]:
assert m_stats["WinRate"].between(0, 1).all()
assert w_stats["WinRate"].between(0, 1).all()

assert m_stats["EFG"].between(0, 1).all()
assert w_stats["EFG"].between(0, 1).all()

assert m_stats["OpponentEFG"].between(0, 1).all()
assert w_stats["OpponentEFG"].between(0, 1).all()

print("Team-season statistic checks passed.")

display(
    m_stats[
        [
            "Season",
            "TeamID",
            "GamesPlayed",
            "Wins",
            "WinRate",
            "AvgScoringMargin",
            "EFG",
            "OpponentEFG",
            "EFGDifferential"
        ]
    ].head(10)
)

Team-season statistic checks passed.


,Season,TeamID,GamesPlayed,Wins,WinRate,AvgScoringMargin,EFG,OpponentEFG,EFGDifferential
0,2003,1102,28,12,0.428571,0.250000,0.579443,0.510522,0.068922
1,2003,1103,27,13,0.481481,0.629630,0.534814,0.545809,-0.010995
2,2003,1104,28,17,0.607143,4.285714,0.475953,0.476190,-0.000238
3,2003,1105,26,7,0.269231,-4.884615,0.457241,0.511089,-0.053848
4,2003,1106,28,13,0.464286,-0.142857,0.479005,0.451505,0.027500
5,2003,1107,28,7,0.250000,-9.785714,0.487259,0.538001,-0.050742
6,2003,1108,33,14,0.424242,-4.030303,0.469040,0.484053,-0.015013
7,2003,1110,30,16,0.533333,4.466667,0.514633,0.486809,0.027824
8,2003,1111,26,16,0.615385,1.538462,0.499404,0.514646,-0.015242
9,2003,1112,28,25,0.892857,14.964286,0.514946,0.456378,0.058568


In [9]:
def process_seeds(seeds):

    seeds = seeds.copy()

    seeds["SeedNum"] = (
        seeds["Seed"]
        .str.extract(r"(\d+)")[0]
        .astype(int)
    )

    return seeds[
        ["Season", "TeamID", "SeedNum"]
    ]

m_seeds_clean = process_seeds(m_seeds)
w_seeds_clean = process_seeds(w_seeds)

display(m_seeds_clean.head())

assert m_seeds_clean["SeedNum"].between(1, 16).all()
assert w_seeds_clean["SeedNum"].between(1, 16).all()

print("Seed checks passed.")


,Season,TeamID,SeedNum
0,1985,1207,1
1,1985,1210,2
2,1985,1228,3
3,1985,1260,4
4,1985,1374,5


Seed checks passed.


In [10]:
def create_tournament_matchups(tourney):

    matchups = pd.DataFrame()

    matchups["Season"] = tourney["Season"]

    matchups["Team1"] = np.minimum(
        tourney["WTeamID"],
        tourney["LTeamID"]
    )

    matchups["Team2"] = np.maximum(
        tourney["WTeamID"],
        tourney["LTeamID"]
    )

    # Target = 1 if lower TeamID won
    # Target = 0 if higher TeamID won
    matchups["Target"] = (
        tourney["WTeamID"] ==
        matchups["Team1"]
    ).astype(int)

    return matchups

m_matchups = create_tournament_matchups(m_tourney)
w_matchups = create_tournament_matchups(w_tourney)

display(m_matchups.head())

,Season,Team1,Team2,Target
0,1985,1116,1234,1
1,1985,1120,1345,1
2,1985,1207,1250,1
3,1985,1229,1425,1
4,1985,1242,1325,1


In [11]:
m_first_detailed_season = m_reg["Season"].min()
w_first_detailed_season = w_reg["Season"].min()

print("First men's detailed season:", m_first_detailed_season)
print("First women's detailed season:", w_first_detailed_season)

m_matchups = m_matchups[
    m_matchups["Season"] >= m_first_detailed_season
].copy()

w_matchups = w_matchups[
    w_matchups["Season"] >= w_first_detailed_season
].copy()

print(
    "Men tournament games with detailed regular-season coverage:",
    len(m_matchups)
)

print(
    "Women tournament games with detailed regular-season coverage:",
    len(w_matchups)
)

First men's detailed season: 2003
First women's detailed season: 2010
Men tournament games with detailed regular-season coverage: 1449
Women tournament games with detailed regular-season coverage: 961


In [12]:
TEAM_STAT_COLUMNS = [
    "Season",
    "TeamID",
    "AvgScoringMargin",
    "WinRate",
    "EFGDifferential"
]

In [13]:
def build_model_dataset(matchups, stats, seeds):

    team_stats = stats[TEAM_STAT_COLUMNS].copy()

    # =========================================================
    # TEAM 1 REGULAR-SEASON STATISTICS
    # =========================================================

    team1_stats = team_stats.rename(columns={
        "TeamID": "Team1",
        "AvgScoringMargin": "Team1_AvgMargin",
        "WinRate": "Team1_WinRate",
        "EFGDifferential": "Team1_EFGDifferential"
    })

    data = matchups.merge(
        team1_stats,
        on=["Season", "Team1"],
        how="inner"
    )

    # =========================================================
    # TEAM 2 REGULAR-SEASON STATISTICS
    # =========================================================

    team2_stats = team_stats.rename(columns={
        "TeamID": "Team2",
        "AvgScoringMargin": "Team2_AvgMargin",
        "WinRate": "Team2_WinRate",
        "EFGDifferential": "Team2_EFGDifferential"
    })

    data = data.merge(
        team2_stats,
        on=["Season", "Team2"],
        how="inner"
    )

    # =========================================================
    # TEAM 1 SEED
    # =========================================================

    team1_seeds = seeds.rename(columns={
        "TeamID": "Team1",
        "SeedNum": "Team1_Seed"
    })

    data = data.merge(
        team1_seeds,
        on=["Season", "Team1"],
        how="inner"
    )

    # =========================================================
    # TEAM 2 SEED
    # =========================================================

    team2_seeds = seeds.rename(columns={
        "TeamID": "Team2",
        "SeedNum": "Team2_Seed"
    })

    data = data.merge(
        team2_seeds,
        on=["Season", "Team2"],
        how="inner"
    )

    # =========================================================
    # CREATE THE FOUR MATCHUP FEATURES
    # =========================================================

    data["SeedDiff"] = (
        data["Team1_Seed"] -
        data["Team2_Seed"]
    )

    data["AvgScoringMarginDiff"] = (
        data["Team1_AvgMargin"] -
        data["Team2_AvgMargin"]
    )

    data["WinRateDiff"] = (
        data["Team1_WinRate"] -
        data["Team2_WinRate"]
    )

    data["EFGDifferentialDiff"] = (
        data["Team1_EFGDifferential"] -
        data["Team2_EFGDifferential"]
    )

    return data

m_model = build_model_dataset(
    m_matchups,
    m_stats,
    m_seeds_clean
)

w_model = build_model_dataset(
    w_matchups,
    w_stats,
    w_seeds_clean
)

In [14]:
FINAL_COLUMNS = [
    "Season",
    "Team1",
    "Team2",

    "SeedDiff",
    "AvgScoringMarginDiff",
    "WinRateDiff",
    "EFGDifferentialDiff",

    "Target"
]

m_processed = m_model[FINAL_COLUMNS].copy()
w_processed = w_model[FINAL_COLUMNS].copy()

In [15]:
print("MEN")
print("Rows:", len(m_processed))
print("Seasons:",
      m_processed["Season"].min(),
      "-",
      m_processed["Season"].max())
print("Missing values:")
print(m_processed.isna().sum())

print("\nTarget distribution:")
print(m_processed["Target"].value_counts(normalize=True))


print("\n------------------------------\n")


print("WOMEN")
print("Rows:", len(w_processed))
print("Seasons:",
      w_processed["Season"].min(),
      "-",
      w_processed["Season"].max())
print("Missing values:")
print(w_processed.isna().sum())

print("\nTarget distribution:")
print(w_processed["Target"].value_counts(normalize=True))

assert m_processed.isna().sum().sum() == 0
assert w_processed.isna().sum().sum() == 0

assert set(m_processed["Target"].unique()).issubset({0, 1})
assert set(w_processed["Target"].unique()).issubset({0, 1})

assert (m_processed["Team1"] < m_processed["Team2"]).all()
assert (w_processed["Team1"] < w_processed["Team2"]).all()

print("Final validation checks passed.")

MEN
Rows: 1449
Seasons: 2003 - 2025
Missing values:
Season                  0
Team1                   0
Team2                   0
SeedDiff                0
AvgScoringMarginDiff    0
WinRateDiff             0
EFGDifferentialDiff     0
Target                  0
dtype: int64

Target distribution:
Target
1    0.500345
0    0.499655
Name: proportion, dtype: float64

------------------------------

WOMEN
Rows: 961
Seasons: 2010 - 2025
Missing values:
Season                  0
Team1                   0
Team2                   0
SeedDiff                0
AvgScoringMarginDiff    0
WinRateDiff             0
EFGDifferentialDiff     0
Target                  0
dtype: int64

Target distribution:
Target
1    0.504683
0    0.495317
Name: proportion, dtype: float64
Final validation checks passed.


In [16]:
display(m_processed.head(10))

,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1
5,2003,1140,1163,7,0.140860,0.041935,-0.022524,0
6,2003,1161,1181,11,-10.233333,-0.233333,0.015743,0
7,2003,1153,1211,-1,-3.049539,-0.134793,-0.036629,0
8,2003,1228,1443,-9,7.355914,0.058065,0.069843,1
9,2003,1242,1429,-13,9.100000,0.033333,0.057274,1


In [17]:
m_processed.to_csv(
    PROCESSED_DIR / "men_processed.csv",
    index=False
)

w_processed.to_csv(
    PROCESSED_DIR / "women_processed.csv",
    index=False
)

print("Saved:")
print(PROCESSED_DIR / "men_processed.csv")
print(PROCESSED_DIR / "women_processed.csv")

Saved:
../data/processed/men_processed.csv
../data/processed/women_processed.csv
